## Cross model generalization tests


Distilled dataset tested : 

| Run Name          | Augs | Accuracy | Time (min) |
|-------------------|------|----------|----------|
| fresh-pond-40     | 2    | 23.22     | 2.41     |
| morning-moon-39   | 6    | 41.08      | 3.01      |
| upbeat-bee-42     | 10   | 41.75     | 2.46      |
| random     | X   | 31    | 2.21     |
| full_data     | X   | 97.52    | 53     |

Results 256x256 (crop 224) : 

| Run Name          | Augs | Accuracy | Time (min) |
|-------------------|------|----------|----------|
| icy-glade-43     | 3    | 60.94     | 7.47     |
| random     | X    | 60.94     | 7.47     |



### TODO
<input type="checkbox"> Run distillation with different models <br>
<input type="checkbox"> Run cross generalization eval <br>
<input type="checkbox"> Run new test with more augmentations <br>
<input type="checkbox"> Add a per class accuracy <br>
<input type="checkbox"> Run ablation study on WeightedSampler (low augs, high augs)<br>

In [ ]:
from itertools import product
import subprocess
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys


In [ ]:
MODELS = ["dinov2_vitb", "clip_vitb", "mocov3_vitb"]
DATASET = "fish4knowledge"
results = {m: {} for m in MODELS}
DISTILL_CONFIGS = {
    "dinov2_vitb": {"syn_res": 224, "real_res": 224, "crop_res": 224, "train_crop_mode": "random", "augs_per_batch": 3, "num_eval": 2},
    "clip_vitb":   {"syn_res": 224, "real_res": 224, "crop_res": 224, "train_crop_mode": "random", "augs_per_batch": 3, "num_eval": 2},
    "mocov3_vitb": {"syn_res": 224, "real_res": 224, "crop_res": 224, "train_crop_mode": "random", "augs_per_batch": 3, "num_eval": 2},
}

### Run distillation for all models

In [ ]:
for model, config in DISTILL_CONFIGS.items():
    print(f"\n{'='*50}")
    print(f"Distilling {model} on {DATASET}")
    print('='*50)

    env = os.environ.copy()
    env["DATASET"] = DATASET
    env["MODEL"] = model

    process = subprocess.Popen(
        ["./run.sh", "distill", f"--augs_per_batch={config['augs_per_batch']}",
         f"--syn_res={config['syn_res']}",
         f"--real_res={config['real_res']}",
         f"--crop_res={config['crop_res']}",
         f"--train_crop_mode={config['train_crop_mode']}",
         f"--run_name={model}_distill_{config['syn_res']}_augs{config['augs_per_batch']}"],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stderr dans stdout
        text=True,
        cwd="/home/alex/internship/GradientDistillation"
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    process.wait()
    print(f"\nReturn code: {process.returncode}")

### Visualize dataset from every model

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils

In [ ]:


for model, config in DISTILL_CONFIGS.items():
    print(f"\n{'='*50}")
    print(f"Visualizing {model} distillation results")
    print('='*50)

    data = torch.load(f"../logged_files/distillation/fish4knowledge/{model}/{model}_distill_{config['syn_res']}_augs{config['augs_per_batch']}/data.pth", weights_only=False)
    images = data["images"]  # (N, C, H, W)
    print(f"Shape: {images.shape}")

    grid = vutils.make_grid(images, nrow=4, padding=2)
    plt.figure(figsize=(12, 4))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.axis("off")
    plt.title(f"Distilled Images — {model}")
    plt.tight_layout()
    plt.show()


### Run cross generalization eval

In [ ]:
MODELS = list(DISTILL_CONFIGS.keys())
RUN_NAMES = {model: f"{model}_distill_{DISTILL_CONFIGS[model]['syn_res']}_augs{DISTILL_CONFIGS[model]['augs_per_batch']}" for model in MODELS}

results = {m: {} for m in MODELS}

for train_model, eval_model in product(MODELS, repeat=2):
    print(f"\n{'='*50}")
    print(f"Train: {train_model} | Eval: {eval_model}")
    print('='*50)

    env = os.environ.copy()
    env["MODEL"] = train_model
    env["EVAL_MODEL"] = eval_model
    env["DATASET"] = DATASET

    result = subprocess.run(
        ["./run.sh", "eval", f"--run_name={RUN_NAMES[train_model]}", f"--num_eval={DISTILL_CONFIGS[eval_model]['num_eval']}"],
        env=env,
        capture_output=True,
        text=True,
        cwd="/home/alex/internship/GradientDistillation"
    )

    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
    print("Return code:", result.returncode)

    match = re.search(r"Top 1 Mean.*?:\s*([\d.]+)\s*\S+\s*([\d.]+)", result.stdout)
    score = float(match.group(1)) if match else float("nan")

    results[train_model][eval_model] = score
    print(f"Score: {score}")

In [ ]:
matrix = np.array([[results[t][e] for e in MODELS] for t in MODELS])

# Affichage
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".3f",
    xticklabels=MODELS,
    yticklabels=MODELS,
    cmap="YlOrRd",
    ax=ax,
    vmin=0,
    vmax=100,
)
ax.set_xlabel("Eval Model", fontsize=12)
ax.set_ylabel("Train Model", fontsize=12)
ax.set_title(f"Cross-Generalization Confusion Matrix\nDataset: {DATASET}", fontsize=14)
plt.tight_layout()
plt.savefig("cross_generalization_matrix.png", dpi=150)
plt.show()

### Random cross generalization eval

In [ ]:
%env MODEL=dinov2_vitb
%env EVAL_MODEL=dinov2_vitb
%env DATASET=fish4knowledge

!./run.sh random

In [ ]:
random_results = {eval_model: {} for eval_model in MODELS}

for eval_model in MODELS:
    print(f"\n{'='*50}")
    print(f"Random baseline | Eval: {eval_model}")
    print('='*50)

    env = os.environ.copy()
    env["MODEL"] = eval_model   # random reals évalués sur eval_model
    env["DATASET"] = DATASET

    result = subprocess.run(
        [
            "./run.sh", "random",
            "--ipc=1",
            "--real_res=224",
            "--crop_res=224",
            "--skip_if_exists=False",
        ],
        env=env,
        capture_output=True,
        text=True,
        cwd="/home/alex/internship/GradientDistillation"
    )

    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
    print("Return code:", result.returncode)

    match = re.search(r"Top1:\s*([\d.]+)", result.stdout)
    score = float(match.group(1)) if match else float("nan")

    random_results[eval_model] = score
    print(f"Random score ({eval_model}): {score}")

In [ ]:
print("\n=== Distilled (diagonal) vs Random baseline ===")
print(f"{'Model':<20} {'Distilled':>10} {'Random':>10} {'Gain':>10}")
print("-" * 52)
for m in MODELS:
    distilled = results[m][m]
    random_score = random_results[m]
    gain = distilled - random_score
    print(f"{m:<20} {distilled:>10.2f} {random_score:>10.2f} {gain:>+10.2f}")